# Error Recovery in Agents

Learning objectives:
- Classify errors (transient, permanent, semantic)
- Implement retry with exponential backoff
- Design fallback strategies
- Validate LLM outputs before using
- Use circuit breakers to prevent cascading failures

**Prerequisites:** Python 3.8+, anthropic SDK

In [ ]:
import time
import json
from datetime import datetime, timedelta
from enum import Enum
from typing import Optional, Any, Callable, List
from anthropic import Anthropic

client = Anthropic()
print("✓ Setup ready: error recovery libraries")

## Level 1: Retry with Exponential Backoff

In [ ]:
def classify_error(error_msg: str) -> str:
    """Classify error as transient, permanent, or semantic"""
    msg = error_msg.lower()
    if any(x in msg for x in ["timeout", "rate limit", "503", "502", "connection"]):
        return "transient"
    elif any(x in msg for x in ["400", "404", "401", "not found"]):
        return "permanent"
    else:
        return "semantic"

def call_with_retry(func: Callable, max_retries: int = 3) -> Optional[Any]:
    """Call with exponential backoff"""
    backoff = 1
    for attempt in range(1, max_retries + 1):
        try:
            return func()
        except Exception as e:
            error_type = classify_error(str(e))
            if error_type == "permanent":
                print(f"Permanent error, not retrying: {e}")
                return None
            if attempt < max_retries:
                print(f"Attempt {attempt}: {error_type} error. Retrying in {backoff}s...")
                time.sleep(backoff)
                backoff *= 2
            else:
                print(f"Max retries exhausted")
                return None

# Test
def flaky_function():
    import random
    if random.random() < 0.7:
        raise TimeoutError("Timeout")
    return "Success"

print("Testing retry logic:")
result = call_with_retry(flaky_function, max_retries=3)
print(f"Result: {result}")

## Level 2: Validation and Self-Correction

In [ ]:
class ValidatingLLMAgent:
    """Validate LLM outputs and ask for correction"""
    
    def __init__(self, client):
        self.client = client
        self.max_corrections = 2
    
    def validate_json(self, text: str) -> tuple[bool, Optional[dict], str]:
        try:
            data = json.loads(text)
            return True, data, ""
        except json.JSONDecodeError as e:
            return False, None, f"Invalid JSON: {str(e)[:100]}"
    
    def query_with_validation(self, prompt: str, validate_type: str = "json") -> Optional[Any]:
        messages = [{"role": "user", "content": prompt}]
        
        for attempt in range(1, self.max_corrections + 1):
            response = self.client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=200,
                messages=messages
            )
            
            output = response.content[0].text
            
            if validate_type == "json":
                is_valid, data, error = self.validate_json(output)
                if is_valid:
                    print(f"✓ Valid JSON on attempt {attempt}")
                    return data
            else:
                return output
            
            if attempt < self.max_corrections:
                print(f"✗ Invalid {validate_type} on attempt {attempt}: {error}")
                messages.append({"role": "assistant", "content": output})
                messages.append({"role": "user", "content": f"Invalid output. Error: {error}. Please fix it."})
            else:
                print(f"✗ Max corrections exhausted")
                return None

# Test
agent = ValidatingLLMAgent(client)
result = agent.query_with_validation(
    'Generate JSON with keys "name" and "age". Output only valid JSON.',
    validate_type="json"
)
print(f"Final result: {result}")

## Level 3: Circuit Breaker Pattern

In [ ]:
class CircuitState(Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"

class CircuitBreaker:
    """Prevent hammering of failing services"""
    
    def __init__(self, failure_threshold: int = 5, recovery_timeout: int = 30):
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.last_failure = None
    
    def record_success(self):
        self.failure_count = 0
        self.state = CircuitState.CLOSED
    
    def record_failure(self):
        self.failure_count += 1
        self.last_failure = datetime.now()
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
    
    def can_call(self) -> bool:
        if self.state == CircuitState.CLOSED:
            return True
        elif self.state == CircuitState.OPEN:
            if datetime.now() > self.last_failure + timedelta(seconds=self.recovery_timeout):
                self.state = CircuitState.HALF_OPEN
                self.failure_count = 0
                return True
            return False
        return True
    
    def status(self) -> str:
        return f"{self.state.value} ({self.failure_count} failures)"

# Test
breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=5)

print("Simulating failures:")
for i in range(10):
    if breaker.can_call():
        if i < 3:  # Fail first 3
            breaker.record_failure()
            print(f"  Attempt {i+1}: FAIL - Circuit: {breaker.status()}")
        else:
            breaker.record_success()
            print(f"  Attempt {i+1}: SUCCESS - Circuit: {breaker.status()}")
    else:
        print(f"  Attempt {i+1}: SKIPPED - Circuit OPEN")
    time.sleep(0.5)

## Key Takeaways

1. **Classify errors correctly** — Transient (retry), permanent (don't), semantic (validate/clarify)
2. **Exponential backoff** — 1s, 2s, 4s, 8s prevents hammering recovering services
3. **Validate before using** — Never trust LLM output without validation
4. **Fallback strategies** — Primary tool → fallback tool → simplified approach
5. **Circuit breakers** — Stop retrying unhealthy services; check recovery periodically
6. **Log everything** — Error type, retry count, final outcome. Data drives improvements.

**Related:** Observability, Latency Optimization, Agent Monitoring

## Error Recovery in Agents: baseline implementation

**Title:** Error Recovery in Agents. **Prerequisites:** Python dictionaries and functions. **Estimated time:** 20 minutes. **Objectives:** trace one bounded operation, compare outcomes, and test an edge case.

The baseline below is a deterministic reference point. It makes one bounded decision so we can compare behavior before adding model-driven choices.

In [ ]:
def baseline_route(task):
    """Small transparent baseline for a bounded task."""
    if not isinstance(task, dict) or not task.get("goal"):
        return {"ok": False, "reason": "invalid_input"}
    return {"ok": True, "action": task.get("action", "answer"), "steps": 1}

example = baseline_route({"goal": "summarize", "action": "retrieve"})
assert example["ok"] and example["steps"] == 1

## Comparison and metric

Compare the baseline with the lesson technique on the same examples. Track completion rate, invalid-action rate, and latency; inspect each failure rather than relying only on an aggregate score.

In [ ]:
cases = [{"goal": "summarize"}, {}, {"goal": "update", "action": "review"}]
results = [baseline_route(case) for case in cases]
completion_rate = sum(result["ok"] for result in results) / len(results)
print({"completion_rate": completion_rate, "invalid_inputs": sum(not r["ok"] for r in results)})
assert results[1]["reason"] == "invalid_input"

## Edge and failure cases

**Failure case:** missing goal data must stop before an action is selected. **Exercise / Your turn:** add an allowlist for actions and a maximum step count, then assert that an unknown action is rejected. Explain how the metric changes.

## Summary, next lesson, and practice

Summary: define the task boundary, validate state, measure end-to-end completion, and stop safely on invalid inputs. Next lesson: apply the same checks to the next curriculum concept. Practice: explain the baseline, its limitation, and the evidence needed to ship an improved design.